# EDA - parte 1: descritivas e a definicao de churn

Objetivo: estatisticas descritivas da base limpa e, principalmente, fixar o N do churn
que a `vw_kpis_mensais` vai usar. As consultas maiores vivem em `sql/consultas/` e sao
lidas daqui, para o SQL ficar versionado e nao escondido em string de notebook.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")
# o driver e psycopg3; SQLAlchemy precisa do esquema explicito na URL
url = os.environ["DATABASE_URL"].replace("postgresql://", "postgresql+psycopg://", 1)
eng = create_engine(url)

def consulta(nome):
    return Path(f"../sql/consultas/{nome}").read_text(encoding="utf-8")

## O que conta como venda?

Antes de qualquer media: as analises estruturais (RFM, coorte, Pareto) precisam de uma
definicao de venda concretizada. Olhando o peso de cada status na base:

In [12]:
pd.read_sql("""
    SELECT status_pedido, COUNT(*) AS itens, COUNT(DISTINCT id_pedido) AS pedidos,
           ROUND(SUM(valor_total), 2) AS faturamento
    FROM dw.vw_vendas
    GROUP BY status_pedido ORDER BY faturamento DESC
""", eng)

,status_pedido,itens,pedidos,faturamento
0,Entregue,5984,3251,5638779.48
1,Cancelado,581,313,543283.68
2,Enviado,439,248,432613.00
3,Devolvido,328,174,378851.95
4,Processando,311,160,308839.32


Decisao: **venda concretizada = Entregue + Enviado**. Cancelado e Devolvido nao sao
receita; Processando ainda pode virar cancelamento, entao fica fora por conservadorismo.
Enviado entra porque a mercadoria saiu e a receita foi reconhecida - da para argumentar
que um Enviado ainda vira Devolvido, mas o volume e pequeno (439 itens) e a alternativa
(so Entregue) descartaria venda legitima em transito. Vai para `docs/decisoes.md`.

In [3]:
desc = pd.read_sql(consulta("q_descritivas.sql"), eng)
desc.round(2)

,metrica,minimo,q1,mediana,media,q3,maximo,desvio
0,valor_unitario,47.42,116.86,195.22,654.34,355.15,41186.73,1350.44
1,quantidade,1.00,1.00,1.00,1.43,2.00,4.00,0.75
2,valor_total_item,47.47,139.95,257.38,945.26,611.84,103544.44,2408.83
3,valor_pedido,47.56,235.71,549.16,1735.18,1877.96,103840.51,3358.50


Tudo que e dinheiro e fortemente assimetrico: em `valor_unitario` a media (654) e mais
de 3x a mediana (195), e o maximo (41.186) esta duas ordens de grandeza acima do tipico.
Sao os outliers de preco que o gerador plantou e que a validacao de proposito nao
quarentenou - outlier legitimo e informacao, nao erro. Essa assimetria ja adianta a
discussao IQR vs z-score da parte 2: media e desvio sao pessimos resumos aqui.
`quantidade` e bem comportada (1 a 4, mediana 1).

In [4]:
mensal = pd.read_sql("""
    SELECT ano_mes, ROUND(SUM(valor_total), 2) AS faturamento,
           COUNT(DISTINCT id_pedido) AS pedidos
    FROM dw.vw_vendas
    WHERE status_pedido IN ('Entregue', 'Enviado')
    GROUP BY ano_mes ORDER BY ano_mes
""", eng)
mensal

,ano_mes,faturamento,pedidos
0,2024-07,187636.21,119
1,2024-08,202990.18,118
2,2024-09,224241.17,113
3,2024-10,195820.24,127
4,2024-11,335323.65,207
5,2024-12,288509.58,163
6,2025-01,185879.29,109
7,2025-02,179800.82,108
8,2025-03,164442.27,118
9,2025-04,208585.90,126


Novembro salta nos dois anos (335k em 2024, 500k em 2025, contra ~200k de mes tipico) -
sazonalidade de Black Friday clara. Tambem ha crescimento de nivel entre 2024/25 e
2025/26. Os dois padroes sao o que a pagina Visao Geral e o modelo do bloco 4 precisam
capturar.

## Fixando o N do churn

O enunciado pede churn rate mas nao define churn. A definicao operacional escolhida:
cliente que ficou mais de N dias sem comprar. O N sai dos dados - distribuicao do
intervalo entre compras consecutivas do mesmo cliente (grao pedido, so venda
concretizada):

In [9]:
pd.read_sql(consulta("q_intervalo_compras.sql"), eng).round(1)

,n_intervalos,minimo,mediana,media,p75,p90,p95,maximo
0,2834,1,21.0,40.8,50.0,101.0,154.0,580


In [10]:
# quantos intervalos cabem em 90 dias?
pd.read_sql("""
    WITH pedidos AS (
        SELECT DISTINCT id_cliente, id_pedido, data_venda
        FROM dw.vw_vendas
        WHERE status_pedido IN ('Entregue', 'Enviado')
    ),
    intervalos AS (
        SELECT data_venda - LAG(data_venda) OVER (
            PARTITION BY id_cliente ORDER BY data_venda) AS dias
        FROM pedidos
    )
    SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE dias <= 90) / COUNT(*), 1)
           AS pct_ate_90_dias
    FROM intervalos WHERE dias IS NOT NULL AND dias > 0
""", eng)

,pct_ate_90_dias
0,87.8


O percentil 90 do intervalo e 101 dias. Arredondo para **N = 90 dias (3 meses)**: os
KPIs sao mensais, entao o corte precisa ser multiplo de mes cheio, e 3 meses e o multiplo
que fica mais perto do p90 - cobre a grande maioria dos ritmos de recompra reais da base.
Quem passa de 90 dias sem comprar esta fora do ritmo de 9 em cada 10 recompras.

O maximo de 580 dias incomoda: e um cliente que voltou depois de quase dois anos. Com
N=90 ele foi dado como churned e depois ressuscitou. Aceitavel - churn aqui e operacional
(cliente esfriou, merece atencao), nao sentenca definitiva. A `vw_kpis_mensais` usa esse
N; a justificativa vai para `docs/decisoes.md`.

## Parte 2: correlacao

Pearson mede relacao linear e sofre com os outliers vistos acima; Spearman rankeia
antes, entao e robusto. Rodando os dois para ver onde divergem. Dois graos: item
(preco x quantidade) e cliente (frequencia x monetario, direto da vw_rfm).

In [11]:
itens = pd.read_sql("""
    SELECT valor_unitario::float, quantidade::float, valor_total::float
    FROM dw.vw_vendas WHERE status_pedido IN ('Entregue', 'Enviado')
""", eng)
print("Pearson:")
print(itens.corr(method="pearson").round(3))
print()
print("Spearman:")
print(itens.corr(method="spearman").round(3))

Pearson:
                valor_unitario  quantidade  valor_total
valor_unitario           1.000       0.013        0.828
quantidade               0.013       1.000        0.239
valor_total              0.828       0.239        1.000

Spearman:
                valor_unitario  quantidade  valor_total
valor_unitario           1.000       0.011        0.903
quantidade               0.011       1.000        0.391
valor_total              0.903       0.391        1.000


Preco x quantidade praticamente nao se correlacionam (cliente nao leva menos unidades
porque o item e caro - quantidade aqui e 1 a 4). O par forte e valor_total x
valor_unitario, e o jeito como os dois coeficientes divergem conta a historia: Spearman
(0.90) acima de Pearson (0.83) diz que a relacao e monotonica mas nao linear - o valor do
item e preco x quantidade, entao a nuvem abre em leques de 1x a 4x o preco, e os outliers
esticam as pontas. Ja em quantidade x valor_total o Pearson (0.24) subestima: a variancia
do total e dominada pelo preco, e so nos ranks (0.39) a quantidade aparece.

In [13]:
clientes = pd.read_sql("""
    SELECT recencia_dias::float, frequencia::float, monetario::float FROM dw.vw_rfm
""", eng)
print("Pearson:")
print(clientes.corr(method="pearson").round(3))
print()
print("Spearman:")
print(clientes.corr(method="spearman").round(3))

Pearson:
               recencia_dias  frequencia  monetario
recencia_dias          1.000      -0.274     -0.263
frequencia            -0.274       1.000      0.844
monetario             -0.263       0.844      1.000

Spearman:
               recencia_dias  frequencia  monetario
recencia_dias          1.000      -0.378     -0.313
frequencia            -0.378       1.000      0.775
monetario             -0.313       0.775      1.000


No grao cliente: frequencia e monetario andam juntos (Pearson 0.84 - quem compra mais
vezes gasta mais, sem surpresa), e recencia e negativamente correlacionada com os dois
(-0.3 a -0.4): cliente frequente tende a ter comprado ha pouco. E a estrutura que o RFM
explora - os tres eixos carregam informacao parcialmente redundante, mas nao identica,
o que justifica manter os tres scores em vez de um so.

## Outliers: IQR vs z-score

Os outliers de preco ficaram no fato de proposito (outlier legitimo e informacao de
negocio, nao erro - decisao registrada no bloco 1). A questao aqui e qual criterio usar
para MARCA-LOS na analise. Comparando os dois por categoria, via `q_outliers.sql`:

In [14]:
pd.read_sql(consulta("q_outliers.sql"), eng)

,categoria,n,q1,q3,limite_iqr,limite_z3,outliers_iqr,outliers_z3
0,Acessorios,1592,94.88,256.10,497.93,616.72,7,7
1,Casa,1127,215.21,415.52,715.99,2208.09,221,5
2,Eletronicos,1040,1301.64,3526.35,6863.42,9752.90,6,6
3,Esporte,1164,115.27,195.67,316.28,561.82,193,6
4,Livros,652,73.38,87.54,108.79,239.30,4,4
5,Moda,848,170.78,234.63,330.41,733.05,7,7


O z-score marca menos pontos que o IQR em toda categoria, e o motivo e circular: media
e desvio sao calculados COM os proprios outliers dentro, entao o limite de 3 desvios e
empurrado para cima justamente pelos pontos que ele deveria pegar (efeito de
mascaramento). O IQR usa quartis, que nao se movem com meia duzia de valores extremos.

**Escolha: IQR.** Em distribuicao assimetrica como esta (media 3x a mediana), z-score
pressupoe uma normalidade que os dados nao tem. Vai para `docs/decisoes.md`.

TODO: se sobrar tempo, olhar os outliers marcados um a um - alguns sao o mesmo produto
em datas de pico, o que sugere preco dinamico e nao erro de digitacao.